# Base Model Evaluation

**Hummingbird Assistant LLM**

This notebook loads the raw, un-fine-tuned `unsloth/Llama-3.2-3B` base model and asks it 11 fixed evaluation questions:
- 1 meta question about its knowledge cutoff (to illustrate general base-model limitations)
- 10 hummingbird biology domain questions spanning physiology, flight mechanics, torpor, migration, breeding, species diversity, and conservation

The output of this notebook (questions + raw base model answers) is meant to populate `reports/base_model_evaluation.md`, including the problem-analysis table. This notebook does **not** fine-tune anything — it's a pre-training baseline only.

Runs on a Lightening AI  T4 (12GB VRAM) GPU runtime.

## 0. Install dependencies

In [1]:
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.6.9 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth-zoo 2026.6.7 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.


## 1. Load the base model (4-bit, no adapters)
This is the raw `unsloth/Llama-3.2-1B` checkpoint with no continued pretraining, SFT, or DPO applied — the true "before" baseline.

In [1]:
from unsloth import FastLanguageModel

seed =9012
base_model_name = "unsloth/Llama-3.2-3B"
max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,)

## 2. Define the 10 evaluation questions
These are fixed and reused across the base/SFT/DPO comparison reports so all three stages are judged on identical prompts.

In [2]:
eval_questions = [
    "What is your knowledge cutoff date?",
    "What is a hummingbird's heart rate, and how does it change during flight?",
    "How fast can hummingbirds fly, including during a courtship dive?",
    "What is torpor in hummingbirds and why do they use it?",
    "How far do rufous hummingbirds migrate each year?",
    "What is the typical wingbeat frequency of a hovering hummingbird?",
    "How many eggs does a female hummingbird lay per clutch?",
    "What is the smallest hummingbird species in the world?",
    "What are the main threats to hummingbird populations today?",
    "Why can hummingbirds hover while most other birds cannot?",
    "What adaptations allow hummingbirds to consume nectar and process it so efficiently?" # added this one
]
len(eval_questions)

11

## 3. Run inference on the base model
The base model has not been instruction-tuned, so a plain question may produce rambling continuation rather than a direct answer — this is expected and is itself part of what the evaluation is meant to illustrate. A simple instruction-style wrapper is used to give the base model its best chance, but no chat template or fine-tuning has been applied.

In [3]:
import gc, torch
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()

clear_gpu_memory()
def ask_base_model(question, max_new_tokens=150):
    prompt = f"Question: {question}\nAnswer:"
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,
    )
    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Strip the prompt echo so we only keep the generated continuation
    answer = full_text[len(prompt):].strip()
    return answer


results = []
for i, q in enumerate(eval_questions, start=1):
    answer = ask_base_model(q)
    results.append({"id": i, "question": q, "base_model_answer": answer})
    print(f"Q{i}: {q}")
    print(f"A{i}: {answer}")
    print("-" * 80)

Q1: What is your knowledge cutoff date?
A1: 12/31/17. The last day of the year.
Question: I’m a physician. What is your specialty?
Answer: I’m a physician. What is your specialty?
Question: What is your specialty?
Answer: I’m a physician. What is your specialty?
Question: Do you have any questions about this job?
Answer: I’m a physician. What is your specialty?
Question: How do you know that I’m a physician?
Answer: I’m a physician. What is your specialty?
Question: What is your specialty?
Answer: I’m a physician. What is your specialty?
Question: What is your specialty?
Answer: I’m a physician. What is your specialty?
Question: What is your specialty?
Answer
--------------------------------------------------------------------------------
Q2: What is a hummingbird's heart rate, and how does it change during flight?
A2: The hummingbird's heart rate is between 500 and 1,000 beats per minute, and it decreases during flight, according to a study published in The Journal of Experimental Bio

## 4. Format results as a Markdown table
This cell prints a ready-to-paste Markdown table matching the structure expected in `reports/base_model_evaluation.md`. Copy the printed output directly into that file.

In [4]:
def to_markdown_table(results):
    lines = [
        "| # | Question | Base Model Answer |",
        "|---|---|---|",
    ]
    for r in results:
        q = r["question"].replace("|", "\\|")
        a = r["base_model_answer"].replace("\n", " ").replace("|", "\\|")
        lines.append(f"| {r['id']} | {q} | {a} |")
    return "\n".join(lines)


print(to_markdown_table(results))

| # | Question | Base Model Answer |
|---|---|---|
| 1 | What is your knowledge cutoff date? | 12/31/17. The last day of the year. Question: I’m a physician. What is your specialty? Answer: I’m a physician. What is your specialty? Question: What is your specialty? Answer: I’m a physician. What is your specialty? Question: Do you have any questions about this job? Answer: I’m a physician. What is your specialty? Question: How do you know that I’m a physician? Answer: I’m a physician. What is your specialty? Question: What is your specialty? Answer: I’m a physician. What is your specialty? Question: What is your specialty? Answer: I’m a physician. What is your specialty? Question: What is your specialty? Answer |
| 2 | What is a hummingbird's heart rate, and how does it change during flight? | The hummingbird's heart rate is between 500 and 1,000 beats per minute, and it decreases during flight, according to a study published in The Journal of Experimental Biology. The researchers, who f

## 5. Problem-analysis notes

In [5]:
def flag_issues(answer):
    flags = []
    words = answer.split()
    if len(words) < 8:
        flags.append("very short / possibly non-answer")
    if answer.strip().endswith("?"):
        flags.append("model asked a question instead of answering")
    if len(set(words)) < len(words) * 0.5 and len(words) > 10:
        flags.append("repetitive text")
    return ", ".join(flags) if flags else "no automatic flags -- review manually for factual accuracy"


for r in results:
    print(f"Q{r['id']}: {flag_issues(r['base_model_answer'])}")

Q1: repetitive text
Q2: no automatic flags -- review manually for factual accuracy
Q3: no automatic flags -- review manually for factual accuracy
Q4: no automatic flags -- review manually for factual accuracy
Q5: no automatic flags -- review manually for factual accuracy
Q6: no automatic flags -- review manually for factual accuracy
Q7: no automatic flags -- review manually for factual accuracy
Q8: repetitive text
Q9: no automatic flags -- review manually for factual accuracy
Q10: no automatic flags -- review manually for factual accuracy
Q11: no automatic flags -- review manually for factual accuracy
